# Calibration vs. conformal efficiency (RQ1 and RQ2)

Tests if better calibration (lower ECE / NLL / Brier) give smaller conformal prediction sets.

Preliminary evidence from the report: the deep ensemble lowers NLL by 26% and ECE by 47% vs. the baseline, yet the set size is unchanged (1.20 vs 1.19 classes at 90% coverage). Here we test whether that holds across **all methods and 5 seeds**.

In [1]:
# Setup: only the pipeline package (+ common for EPOCHS / DEVICE)
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import Markdown, display

from pipeline import (Baseline, MCDropout, LaplaceLL, SNGP, DeepEnsemble, REPO,
                      train_with_noise, evaluate_conformal, compare, summarize)
from pipeline.evaluate import to_markdown
import common

print("device:", common.DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA not available - training will fall back to", common.DEVICE,
          "and be MUCH slower. In Colab: Runtime > Change runtime type > GPU.")

device: cuda
GPU: NVIDIA GeForce RTX 3070


In [2]:
# ------------------------------- Config ---------------------------------------
SMOKE = False           # True: tiny subsets + 1 epoch to check everything runs end-to-end
EPOCHS = 1 if SMOKE else common.EPOCHS          # 40 = the shared recipe
SEEDS = [0, 1, 2, 3, 4]
NOISE_RATIOS = [0.0]    # clean CIFAR-10 only; the noise study is Problem 1
ALPHA = 0.1             # target coverage 90% (the alpha evaluate_conformal is run at)

# Grid for the set-size-vs-alpha curve (must contain ALPHA). AUC is normalised by the alpha
# range, so it reads as "mean set size over alpha in [0.02, 0.30]". Each extra alpha costs one
# more full inference pass per run (evaluate_conformal re-predicts), so keep the grid modest;
# MC Dropout (T=50) and Laplace (CPU) are the slow ones.
ALPHAS = np.round(np.linspace(0.02, 0.30, 15), 4)
assert ALPHA in ALPHAS
SCORES = ("aps_rand", "lac")

OUT_DIR = REPO / "results" / "calibration_efficiency"   # this notebook's own outputs
OUT_DIR.mkdir(parents=True, exist_ok=True)

# SAME configs as noise_testing.ipynb so existing clean checkpoints are reused
# (train_with_noise raises if a checkpoint's model config differs).
MODELS = {
    "baseline":      Baseline,
    "mc_dropout":    lambda: MCDropout(p=0.5, T=50),
    "laplace":       LaplaceLL,
    "sngp":          lambda: SNGP(norm_bound=6.0),
    "deep_ensemble": lambda: DeepEnsemble(n=5),
}
RUN_MODELS = ["baseline", "mc_dropout", "laplace", "sngp", "deep_ensemble"]

CALIB_KEYS = ["ece", "nll", "brier"]
EFF_KEYS = ["aps_set_size", "aps_auc", "lac_set_size", "lac_auc"]   # *_set_size is at ALPHA
ALL_KEYS = ["accuracy"] + CALIB_KEYS + EFF_KEYS

## 1. Train + evaluate every (method, seed), at every α

For each (method, seed): `train_with_noise` trains the model, then `evaluate_conformal` is called at α = 0.1 (saves `{tag}.json` / `{tag}.npz` to `results/noise_testing/`) and again at every other α in `ALPHAS` to build the set-size-vs-α curve.

Training is 5 methods × 5 seeds, and the ensemble trains 5 networks per run, so 45 ResNet-20s at 40 epochs each.

In [3]:
results = []     # evaluate_conformal result at ALPHA, one per (method, seed)
curves = []      # {"aps_rand": {"sizes": [...], "coverage": [...]}, "lac": ...} per run, aligned with results
i_a = int(np.argmin(np.abs(ALPHAS - ALPHA)))

for name in RUN_MODELS:
    print(f"==================== {name}")
    for s in SEEDS:
        run = train_with_noise(MODELS[name](), noise_ratio=NOISE_RATIOS[0], seed=s,
                               epochs=EPOCHS, smoke=SMOKE)
        model, L = run["model"], run["loaders"]

        # Send the model through the conformal layer
        res = evaluate_conformal(model, L["cal"], L["test"], alpha=ALPHA, seed=s, n_resplits=0,
                                 tag=run["tag"], config=run["config"])
        results.append(res)

        # set-size-vs-alpha curve: same function, other alphas, nothing saved
        cur = {sc: {"sizes": [], "coverage": []} for sc in SCORES}
        for j, a in enumerate(ALPHAS):
            r = res if j == i_a else evaluate_conformal(model, L["cal"], L["test"], alpha=float(a),
                                                        seed=s, n_resplits=0, tag=run["tag"],
                                                        config=run["config"], save=False)
            for sc in SCORES:
                cur[sc]["sizes"].append(r["conformal"][sc]["avg_set_size"])
                cur[sc]["coverage"].append(r["conformal"][sc]["coverage"])
        curves.append(cur)
        print(f"[{run['tag']}] done: APS size@{ALPHA}={res['conformal']['aps_rand']['avg_set_size']:.3f}")

        del run, model, L
        if torch.cuda.is_available():
            torch.cuda.empty_cache()        # release GPU memory (ensembles hold 5 nets)
print(f"\n{len(results)} runs")

==================== baseline
[baseline_noise0_seed0] epoch   1/40  loss=1.5311 acc=0.4300 (cifar 0.4300, svhn nan)  cal_loss=1.8280 cal_acc=0.4162  0.2min
[baseline_noise0_seed0] epoch   2/40  loss=1.0833 acc=0.6112 (cifar 0.6112, svhn nan)  cal_loss=1.1203 cal_acc=0.6022  0.4min
[baseline_noise0_seed0] epoch   3/40  loss=0.8930 acc=0.6842 (cifar 0.6842, svhn nan)  cal_loss=1.0380 cal_acc=0.6292  0.5min
[baseline_noise0_seed0] epoch   4/40  loss=0.7872 acc=0.7247 (cifar 0.7247, svhn nan)  cal_loss=0.9893 cal_acc=0.6626  0.6min
[baseline_noise0_seed0] epoch   5/40  loss=0.7207 acc=0.7518 (cifar 0.7518, svhn nan)  cal_loss=1.1348 cal_acc=0.6362  0.7min
[baseline_noise0_seed0] epoch   6/40  loss=0.6707 acc=0.7679 (cifar 0.7679, svhn nan)  cal_loss=0.8574 cal_acc=0.7181  0.8min
[baseline_noise0_seed0] epoch   7/40  loss=0.6386 acc=0.7808 (cifar 0.7808, svhn nan)  cal_loss=0.8533 cal_acc=0.7182  0.9min
[baseline_noise0_seed0] epoch   8/40  loss=0.6074 acc=0.7889 (cifar 0.7889, svhn nan)  c

D:\University\Fourth Year - 2026-2027\CISC 473\CISC-473-P23\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
D:\University\Fourth Year - 2026-2027\CISC 473\CISC-473-P23\.venv\Lib\site-packages\laplace\baselaplace.py:470: UserWarning: By default `link_approx` is `probit`. Make sure to set it equals to the way you want to call `la(test_data, pred_type=..., link_approx=...)`.
  warnings.warn(


[laplace_noise0_seed0] post-hoc fit done in 77.0s
[laplace_noise0_seed0] done: APS size@0.1=1.215
[laplace_noise0_seed1] epoch   1/40  loss=1.5376 acc=0.4242 (cifar 0.4242, svhn nan)  cal_loss=1.3679 cal_acc=0.4972  0.1min
[laplace_noise0_seed1] epoch   2/40  loss=1.0807 acc=0.6118 (cifar 0.6118, svhn nan)  cal_loss=1.3272 cal_acc=0.5554  0.2min
[laplace_noise0_seed1] epoch   3/40  loss=0.8927 acc=0.6849 (cifar 0.6849, svhn nan)  cal_loss=0.8462 cal_acc=0.7079  0.3min
[laplace_noise0_seed1] epoch   4/40  loss=0.7921 acc=0.7230 (cifar 0.7230, svhn nan)  cal_loss=0.8591 cal_acc=0.7073  0.4min
[laplace_noise0_seed1] epoch   5/40  loss=0.7196 acc=0.7505 (cifar 0.7505, svhn nan)  cal_loss=0.7948 cal_acc=0.7318  0.6min
[laplace_noise0_seed1] epoch   6/40  loss=0.6704 acc=0.7671 (cifar 0.7671, svhn nan)  cal_loss=0.9423 cal_acc=0.6859  0.7min
[laplace_noise0_seed1] epoch   7/40  loss=0.6412 acc=0.7804 (cifar 0.7804, svhn nan)  cal_loss=0.6803 cal_acc=0.7679  0.8min
[laplace_noise0_seed1] epoc

In [4]:
# One flat row per run. NOTE: path is set so this does NOT overwrite the noise notebook's
# results/noise_testing/results.json (compare() writes there by default).
rows = compare(results, path=OUT_DIR / "runs_full_results.json")
display(Markdown(to_markdown(rows, ["model", "seed", "accuracy", "nll", "ece", "brier",
                                    "aps_coverage", "aps_set_size", "lac_coverage", "lac_set_size"])))

| model | seed | accuracy | nll | ece | brier | aps_coverage | aps_set_size | lac_coverage | lac_set_size |
|---|---|---|---|---|---|---|---|---|---|
| baseline | 0 | 0.9067 | 0.2925 | 0.0287 | 0.1391 | 0.8963 | 1.1625 | 0.9038 | 0.9904 |
| baseline | 1 | 0.9022 | 0.2959 | 0.0311 | 0.1430 | 0.9039 | 1.1936 | 0.9032 | 0.9999 |
| baseline | 2 | 0.9118 | 0.2879 | 0.0225 | 0.1352 | 0.9016 | 1.1772 | 0.9086 | 0.9935 |
| baseline | 3 | 0.9074 | 0.2874 | 0.0263 | 0.1373 | 0.8995 | 1.1727 | 0.9011 | 0.9808 |
| baseline | 4 | 0.9062 | 0.2954 | 0.0279 | 0.1407 | 0.8962 | 1.1815 | 0.9041 | 0.9920 |
| mc_dropout | 0 | 0.9022 | 0.2940 | 0.0079 | 0.1425 | 0.9001 | 1.2910 | 0.9032 | 0.9990 |
| mc_dropout | 1 | 0.9002 | 0.2999 | 0.0125 | 0.1454 | 0.8996 | 1.2809 | 0.9070 | 1.0157 |
| mc_dropout | 2 | 0.9026 | 0.3005 | 0.0125 | 0.1439 | 0.8994 | 1.2855 | 0.9070 | 1.0068 |
| mc_dropout | 3 | 0.8993 | 0.3024 | 0.0143 | 0.1459 | 0.8889 | 1.2477 | 0.8999 | 0.9964 |
| mc_dropout | 4 | 0.8992 | 0.3001 | 0.0104 | 0.1464 | 0.8975 | 1.2960 | 0.9017 | 1.0015 |
| laplace | 0 | 0.9074 | 0.2749 | 0.0131 | 0.1358 | 0.8998 | 1.2151 | 0.9076 | 0.9982 |
| laplace | 1 | 0.9069 | 0.2734 | 0.0132 | 0.1346 | 0.8981 | 1.1987 | 0.9035 | 0.9868 |
| laplace | 2 | 0.9024 | 0.2952 | 0.0133 | 0.1435 | 0.8978 | 1.2282 | 0.8985 | 0.9871 |
| laplace | 3 | 0.9058 | 0.2791 | 0.0119 | 0.1369 | 0.8961 | 1.2080 | 0.9044 | 0.9932 |
| laplace | 4 | 0.9058 | 0.2789 | 0.0132 | 0.1361 | 0.8964 | 1.2069 | 0.8992 | 0.9832 |
| sngp | 0 | 0.9019 | 0.3326 | 0.0254 | 0.1471 | 0.8988 | 1.3278 | 0.9077 | 1.0137 |
| sngp | 1 | 0.8920 | 0.3536 | 0.0300 | 0.1581 | 0.8988 | 1.3466 | 0.9031 | 1.0238 |
| sngp | 2 | 0.8982 | 0.3404 | 0.0309 | 0.1518 | 0.9001 | 1.3397 | 0.9033 | 1.0118 |
| sngp | 3 | 0.8978 | 0.3410 | 0.0291 | 0.1524 | 0.8961 | 1.3199 | 0.9005 | 1.0081 |
| sngp | 4 | 0.9020 | 0.3220 | 0.0275 | 0.1452 | 0.8968 | 1.2741 | 0.9000 | 0.9935 |
| deep_ensemble | 0 | 0.9242 | 0.2221 | 0.0139 | 0.1093 | 0.8958 | 1.1757 | 0.9044 | 0.9553 |
| deep_ensemble | 1 | 0.9273 | 0.2210 | 0.0165 | 0.1085 | 0.8950 | 1.1729 | 0.9046 | 0.9519 |
| deep_ensemble | 2 | 0.9253 | 0.2208 | 0.0145 | 0.1091 | 0.8997 | 1.1841 | 0.9027 | 0.9493 |
| deep_ensemble | 3 | 0.9258 | 0.2193 | 0.0158 | 0.1079 | 0.8962 | 1.1843 | 0.9057 | 0.9531 |
| deep_ensemble | 4 | 0.9251 | 0.2229 | 0.0163 | 0.1100 | 0.8958 | 1.1846 | 0.9012 | 0.9483 |

## 2. Threshold-free efficiency: area under the set-size-vs-α curve

Finding the **mean set size over the α grid** for each run.

In [5]:
_trapz = getattr(np, "trapezoid", None) or np.trapz     # np.trapz was renamed in NumPy 2

for row, cur in zip(rows, curves):
    row["aps_auc"] = float(_trapz(cur["aps_rand"]["sizes"], ALPHAS) / (ALPHAS[-1] - ALPHAS[0]))
    row["lac_auc"] = float(_trapz(cur["lac"]["sizes"], ALPHAS) / (ALPHAS[-1] - ALPHAS[0]))
print("AUC added to", len(rows), "runs")

AUC added to 25 runs


In [6]:
# Clean vs. method summary: mean ± std over seeds (summarize() from evaluate.py)
summary = summarize(rows)
display(Markdown(to_markdown(summary, ["model", "n_seeds", "accuracy", "nll", "ece", "brier",
                                       "aps_coverage", "aps_set_size", "lac_coverage", "lac_set_size"])))

# AUC summary in the same format
auc_summary = []
for m in RUN_MODELS:
    g = [r for r in rows if r["model"] == m]
    d = {"model": m, "n_seeds": len(g)}
    for k in ("aps_auc", "lac_auc"):
        v = np.array([r[k] for r in g]); d[k] = float(v.mean()); d[k + "_std"] = float(v.std(ddof=1)) if len(v) > 1 else 0.0
    auc_summary.append(d)
display(Markdown(to_markdown(auc_summary)))

| model | n_seeds | accuracy | nll | ece | brier | aps_coverage | aps_set_size | lac_coverage | lac_set_size |
|---|---|---|---|---|---|---|---|---|---|
| baseline | 5 | 0.9069 ± 0.0034 | 0.2918 ± 0.0040 | 0.0273 ± 0.0032 | 0.1391 ± 0.0030 | 0.8995 ± 0.0034 | 1.1775 ± 0.0114 | 0.9042 ± 0.0027 | 0.9913 ± 0.0069 |
| mc_dropout | 5 | 0.9007 ± 0.0016 | 0.2994 ± 0.0032 | 0.0115 ± 0.0024 | 0.1448 ± 0.0016 | 0.8971 ± 0.0047 | 1.2802 ± 0.0191 | 0.9037 ± 0.0032 | 1.0039 ± 0.0077 |
| laplace | 5 | 0.9057 ± 0.0020 | 0.2803 ± 0.0087 | 0.0130 ± 0.0006 | 0.1374 ± 0.0035 | 0.8976 ± 0.0015 | 1.2114 ± 0.0111 | 0.9027 ± 0.0038 | 0.9897 ± 0.0060 |
| sngp | 5 | 0.8984 ± 0.0041 | 0.3379 ± 0.0117 | 0.0286 ± 0.0022 | 0.1509 ± 0.0051 | 0.8981 ± 0.0017 | 1.3216 ± 0.0285 | 0.9029 ± 0.0031 | 1.0102 ± 0.0110 |
| deep_ensemble | 5 | 0.9255 ± 0.0012 | 0.2212 ± 0.0014 | 0.0154 ± 0.0012 | 0.1090 ± 0.0008 | 0.8965 ± 0.0018 | 1.1803 ± 0.0056 | 0.9037 ± 0.0018 | 0.9516 ± 0.0028 |

| model | n_seeds | aps_auc | lac_auc |
|---|---|---|---|
| baseline | 5 | 1.0866 ± 0.0072 | 0.9265 ± 0.0051 |
| mc_dropout | 5 | 1.1798 ± 0.0150 | 0.9383 ± 0.0032 |
| laplace | 5 | 1.1158 ± 0.0110 | 0.9259 ± 0.0034 |
| sngp | 5 | 1.2109 ± 0.0240 | 0.9507 ± 0.0081 |
| deep_ensemble | 5 | 1.0839 ± 0.0039 | 0.8926 ± 0.0006 |

## 3. Rank Correlations.

ρ near 0 means calibration ranking does not predict efficiency ranking. A positive ρ (worse calibration iff larger sets) means it does. We show results in the following categories:

- **pooled:** all (method, seed) runs together, n = 25
- **within_method:** each metric with the mean removed per method, then pooled. Tests if better-calibrated seeds of the same method get smaller sets.
- **method_means:** the five per-method means
- **partial_acc:** pooled Spearman after regressing ranks on accuracy, i.e. what calibration tells us about set size beyond accuracy

In [7]:
def col(rs, k): return np.array([r[k] for r in rs], dtype=float)

def spearman(x, y):
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0: return float("nan"), float("nan")
    r, p = stats.spearmanr(x, y); return float(r), float(p)

def kendall(x, y):
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0: return float("nan"), float("nan")
    r, p = stats.kendalltau(x, y); return float(r), float(p)

def partial_spearman(x, y, z):
    # Spearman of x, y after linearly removing the rank of z from each rank.
    rx, ry, rz = (stats.rankdata(v) for v in (x, y, z))
    A = np.column_stack([np.ones_like(rz), rz])
    res = lambda r: r - A @ np.linalg.lstsq(A, r, rcond=None)[0]
    ex, ey = res(rx), res(ry)
    if np.std(ex) == 0 or np.std(ey) == 0: return float("nan")
    return float(np.corrcoef(ex, ey)[0, 1])

def boot_ci(x, y, n_boot=2000, seed=0):
    # Bootstrap 95% CI of Spearman, resampling runs.
    rng, rs, n = np.random.RandomState(seed), [], len(x)
    for _ in range(n_boot):
        i = rng.randint(0, n, n)
        if np.std(x[i]) > 0 and np.std(y[i]) > 0:
            rs.append(stats.spearmanr(x[i], y[i])[0])
    return [float(np.percentile(rs, 2.5)), float(np.percentile(rs, 97.5))] if rs else [np.nan, np.nan]


def correlations(rows):
    methods = list(dict.fromkeys(r["model"] for r in rows))
    out = {"pooled": {}, "within_method": {}, "method_means": {}, "partial_acc": {}}

    dem = {k: np.zeros(len(rows)) for k in ALL_KEYS}          # per-method de-meaned metrics
    for m in methods:
        idx = [i for i, r in enumerate(rows) if r["model"] == m]
        for k in ALL_KEYS:
            v = col(rows, k); dem[k][idx] = v[idx] - v[idx].mean()
    mean_rows = [{"model": m, **{k: float(np.mean([r[k] for r in rows if r["model"] == m]))
                                 for k in ALL_KEYS}} for m in methods]

    for c in CALIB_KEYS:
        for e in EFF_KEYS:
            key = f"{c}|{e}"
            x, y = col(rows, c), col(rows, e)
            rs, ps = spearman(x, y); rk, pk = kendall(x, y)
            out["pooled"][key] = {"n": len(x), "spearman": rs, "spearman_p": ps,
                                  "spearman_ci95": boot_ci(x, y), "kendall": rk, "kendall_p": pk}
            rs, ps = spearman(dem[c], dem[e])
            out["within_method"][key] = {"n": len(x), "spearman": rs, "spearman_p": ps}
            rs, ps = spearman(col(mean_rows, c), col(mean_rows, e))
            out["method_means"][key] = {"n": len(mean_rows), "spearman": rs, "spearman_p": ps}
            out["partial_acc"][key] = {"n": len(x), "spearman": partial_spearman(x, y, col(rows, "accuracy"))}
    out["accuracy_vs_eff"] = {e: dict(zip(("spearman", "spearman_p"), spearman(col(rows, "accuracy"), col(rows, e))))
                              for e in EFF_KEYS}
    return out, mean_rows

corr, mean_rows = correlations(rows)

In [12]:
# Report-style comparison: ensemble vs baseline (the Table VII claim), mean over seeds
b = {k: col([r for r in rows if r["model"] == "baseline"], k).mean() for k in ALL_KEYS}
e = {k: col([r for r in rows if r["model"] == "deep_ensemble"], k).mean() for k in ALL_KEYS}
for k in CALIB_KEYS:
    print(f"{k:6s}: ensemble vs baseline {100 * (e[k] - b[k]) / b[k]:+.1f}%")
for k in ["aps_set_size", "lac_set_size", "aps_auc", "lac_auc"]:
    print(f"{k:13s}: baseline {b[k]:.3f}  ensemble {e[k]:.3f}  ({100 * (e[k] - b[k]) / b[k]:+.1f}%)")

ece   : ensemble vs baseline -43.6%
nll   : ensemble vs baseline -24.2%
brier : ensemble vs baseline -21.6%
aps_set_size : baseline 1.178  ensemble 1.180  (+0.2%)
lac_set_size : baseline 0.991  ensemble 0.952  (-4.0%)
aps_auc      : baseline 1.087  ensemble 1.084  (-0.2%)
lac_auc      : baseline 0.927  ensemble 0.893  (-3.7%)


In [13]:
# Save the analysis (per-run evaluate_conformal JSON/NPZ are already in results/noise_testing/)
suffix = "_smoke" if SMOKE else ""
with open(OUT_DIR / f"calibration_efficiency{suffix}.json", "w") as f:
    json.dump({"alphas": ALPHAS.tolist(), "seeds": SEEDS, "rows": rows,
               "curves": curves, "correlations": corr}, f, indent=2)
print("saved to", OUT_DIR / f"calibration_efficiency{suffix}.json")

saved to D:\University\Fourth Year - 2026-2027\CISC 473\CISC-473-P23\results\calibration_efficiency\calibration_efficiency.json
